# 02 — Agent Behavior & Simulation Analysis
Run 1 simulated day, analyze traffic patterns, diurnal energy curves, and anomaly events.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict

from src.simulation_engine.city_model import CityModel
from src.anomaly_detection.detectors import AlertManager, ResourceAnomalyDetector

cfg = {'simulation': {'num_vehicles':100,'num_pedestrians':200,'num_resource_nodes':5,
                       'synthetic_nodes':30,'default_green_ns':30}}
model = CityModel(cfg, seed=42)
alert_mgr = AlertManager()
res_det   = ResourceAnomalyDetector()
print('Running 1 simulated day (1440 ticks)...')

metrics_log = []
alerts_log  = []
for tick in range(1440):
    snap   = model.tick_step()
    alerts = alert_mgr.process_snapshot(snap, resource_detector=res_det)
    metrics_log.append(snap['metrics'].copy())
    alerts_log.extend(alerts)

df = pd.DataFrame(metrics_log)
print(f'Done. Alerts generated: {len(alerts_log)}')
df.head()

In [ ]:
# Diurnal traffic pattern
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
hours = df['hour_of_day'].values

# 1. Vehicles moving vs hour
axes[0,0].fill_between(hours, df['vehicles_moving'], alpha=0.4, color='#2196F3')
axes[0,0].plot(hours, df['vehicles_moving'], color='#1565C0', lw=1.5)
axes[0,0].set_xlabel('Hour of Day'); axes[0,0].set_ylabel('Vehicles Moving')
axes[0,0].set_title('Active Vehicles (Diurnal Pattern)')
axes[0,0].axvspan(7, 9, alpha=0.1, color='red', label='AM Rush')
axes[0,0].axvspan(16, 19, alpha=0.1, color='orange', label='PM Rush')
axes[0,0].legend(fontsize=9)

# 2. Average speed vs hour
axes[0,1].plot(hours, df['avg_vehicle_speed'], color='#4CAF50', lw=1.5)
axes[0,1].fill_between(hours, df['avg_vehicle_speed'], alpha=0.3, color='#4CAF50')
axes[0,1].set_xlabel('Hour of Day'); axes[0,1].set_ylabel('Avg Speed (km/h)')
axes[0,1].set_title('Average Vehicle Speed')

# 3. Energy demand
axes[1,0].plot(hours, df['total_energy_demand_kwh'], color='#FF9800', lw=1.5)
axes[1,0].fill_between(hours, df['total_energy_demand_kwh'], alpha=0.3, color='#FF9800')
axes[1,0].set_xlabel('Hour of Day'); axes[1,0].set_ylabel('Energy Demand (kWh)')
axes[1,0].set_title('Total City Energy Demand')

# 4. Water demand
axes[1,1].plot(hours, df['total_water_demand_m3h'], color='#00BCD4', lw=1.5)
axes[1,1].fill_between(hours, df['total_water_demand_m3h'], alpha=0.3, color='#00BCD4')
axes[1,1].set_xlabel('Hour of Day'); axes[1,1].set_ylabel('Water Demand (m³/h)')
axes[1,1].set_title('Total City Water Demand')

for ax in axes.flat:
    ax.grid(True, alpha=0.3); ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 4))

plt.suptitle('Urban Digital Twin — 24-Hour City Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/simulation_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Alert analysis
print(f'Total alerts in 24h: {len(alerts_log)}')
if alerts_log:
    alert_df = pd.DataFrame(alerts_log)
    print('\nAlert breakdown by type:')
    print(alert_df.groupby(['alert_type','severity']).size().reset_index(name='count').to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Alerts by hour
    alert_df['hour'] = alert_df['tick'] // 60
    hourly = alert_df.groupby('hour').size()
    axes[0].bar(hourly.index, hourly.values, color='#f44336', alpha=0.8)
    axes[0].set_xlabel('Hour'); axes[0].set_ylabel('Alert Count')
    axes[0].set_title('Alerts by Hour of Day')

    # Alerts by severity
    sev_counts = alert_df['severity'].value_counts()
    colors = {'CRITICAL':'#b71c1c','HIGH':'#f44336','MEDIUM':'#FF9800','LOW':'#4CAF50'}
    axes[1].pie(sev_counts.values,
                labels=sev_counts.index,
                colors=[colors.get(s,'#888') for s in sev_counts.index],
                autopct='%1.0f%%', startangle=90)
    axes[1].set_title('Alert Severity Distribution')
    plt.tight_layout()
    plt.savefig('../data/processed/alert_analysis.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No alerts in this run (normal — run longer for statistical anomalies)')

In [ ]:
# Scenario comparison — clear vs rain
cfg2 = {'simulation': {'num_vehicles':50,'num_pedestrians':100,'num_resource_nodes':3,
                        'synthetic_nodes':20,'default_green_ns':30}}

def run_scenario(scenario_type=None, params=None, ticks=180, seed=99):
    m = CityModel(cfg2, seed=seed)
    if scenario_type: m.apply_scenario(scenario_type, params or {})
    speeds, energy = [], []
    for _ in range(ticks):
        s = m.tick_step()
        mt = s['metrics']
        if mt['avg_vehicle_speed'] > 0:
            speeds.append(mt['avg_vehicle_speed'])
        energy.append(mt['total_energy_demand_kwh'])
    return np.array(speeds), np.array(energy)

spd_clear, e_clear = run_scenario()
spd_rain,  e_rain  = run_scenario('rain', {'intensity':'heavy'})
spd_hot,   e_hot   = run_scenario('rain', {'intensity':'light'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scenarios = {'Clear': spd_clear, 'Light Rain': spd_hot, 'Heavy Rain': spd_rain}
axes[0].boxplot(list(scenarios.values()), labels=list(scenarios.keys()),
                patch_artist=True,
                boxprops=dict(facecolor='#E3F2FD'),
                medianprops=dict(color='#1565C0', linewidth=2))
axes[0].set_ylabel('Vehicle Speed (km/h)')
axes[0].set_title('Speed Distribution by Weather Scenario')
axes[0].grid(True, alpha=0.3)

for label, e in [('Clear', e_clear), ('Light Rain', e_hot), ('Heavy Rain', e_rain)]:
    axes[1].plot(e, label=label, lw=1.5)
axes[1].set_xlabel('Tick'); axes[1].set_ylabel('Energy Demand (kWh)')
axes[1].set_title('Energy Demand: Clear vs Rain Scenarios')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('What-If Scenario Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/scenario_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Clear avg speed:      {spd_clear.mean():.1f} km/h')
print(f'Heavy rain avg speed: {spd_rain.mean():.1f} km/h')
print(f'Speed reduction:      {(1-spd_rain.mean()/spd_clear.mean())*100:.1f}%')